In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()
# Create map
Map = geemap.Map(center=[8.5, -80], zoom=7)

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")

panama = countries.filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

roi = panama.geometry()

# Sentinel-2 cloud mask
def mask_s2_clouds(image):

    qa = image.select("QA60")

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    return image.updateMask(mask).divide(10000)

# Sentinel-2 collection
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate("2024-01-01", "2025-05-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    .map(mask_s2_clouds)
)

# Composite clipped to Panama
image = s2.median().clip(roi)

# Visualization parameters
rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 0.3,
}

# Load DEM and clip to Panama
dem = ee.Image("MERIT/DEM/v1_0_3").clip(roi)

# Elevation visualization
dem_vis = {
    "min": 0,
    "max": 3500,
    "palette": [
        "#000000",
        "#478fcd",
        "#86c58e",
        "#afc35e",
        "#8f7131",
        "#b78d4f",
        "#e2b8a6",
        "#ffffff"   # high mountains
    ],
}

# Add layers
Map.addLayer(image, rgb_vis, "Sentinel-2")

Map.addLayer(dem, dem_vis, "Elevation")

# Center map
Map.centerObject(roi, 10)

# Display map
Map

Map(center=[8.515838945899919, -80.10966640141515], controls=(WidgetControl(options=['position', 'transparent_…